# 7) Uncertainty

# 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import math, pickle
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import seaborn as sns
import glob
import os, sys, gc
from pathlib import Path
import ast
from scipy.signal import find_peaks
from typing import Dict, List, Tuple

In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

In [ ]:
from utils import eval_util_module 
import importlib
importlib.reload(eval_util_module)

from utils import utils 
importlib.reload(utils)

# 2. Bootstrap

In [ ]:
df = pd.read_parquet('3_output/5_final_herd_detrend_df_full_cow_weight.gzip')

In [ ]:
# Create dictionary: {cow_id: DataFrame of all records for that cow}
df_groups = {}
for cow_id in tqdm(df['id'].unique(), desc="Creating cow lookup dictionary"):
    df_groups[cow_id] = df[df['id'] == cow_id].copy()

# grouped:
grouped = df.groupby(['GEOID'])['id'].unique()

base_seed= 40

for i in np.arange(1,26,1):
    print('boot_num :', i)
    rng = np.random.default_rng(seed=base_seed + i)  # Different seed per replicate

    # Step 1: Sample cows with replacement per county
    boot_cows = np.concatenate([
        rng.choice(cows[1], size=len(cows[1]), replace=True)
        for cows in grouped.items()
    ])

    # Step 2: Retrieve and concatenate their data
    boot_df = pd.concat([df_groups[cow] for cow in tqdm(boot_cows, desc=f"Building bootstrap {1}", mininterval=300)],
                        ignore_index=True)

    # Step 3: Save the bootstrapped dataset
    boot_df.to_parquet(f'1_data/bootstrap_seed_{base_seed+i}_df.gzip', compression='gzip')

    print(f"Saved bootstrap {i} with {boot_df.shape[0]:,} rows ({boot_df['id'].nunique():,} unique cows)")


# 3. Weighted_ALE

In [ ]:
## US dairy population structure is fixed, so don't calculate the weights per bootstrap which brings more uncertainty (e.g., population structure vairability) besides cow-level response, model fitting, and which specific cows are sampled.

In [ ]:
param = {'control':['lon','lat','month_cos','month_sin','lac_dim']}
control_var = param['control']
feat_var = ['tmin','tmax_ssrd','rh_am','ag_wind_2m']
sub_cols = control_var + feat_var
target = 'herd_milk_resid'
# feat = 'nightrham'


In [ ]:
PAD = 1e-6

quantiles = np.concatenate([[0.0,0.01,0.03],np.arange(0.05,0.95,0.03),[0.95,0.97,0.99,1.0]])
print(quantiles)
mid_quant = (quantiles[:-1] + quantiles[1:]) / 2  # Midpoints of quantile bins

# Variables that don't use quantile-based binning
non_quantile_vars = ['month_cos', 'month_sin', 'lac_dim', 'lat', 'lon']

In [ ]:
base_seed = 40
ale_df = pd.DataFrame()

for i in np.arange(1,26,1):
    print('#' * 10)
    print('seed_num :', base_seed+i)
    
    df = pd.read_parquet(f'1_data/bootstrap_seed_{base_seed+i}_df.gzip')
    
    ## fitting the model:
    best_params = {
            'seed': base_seed+i,
            'objective': 'reg:squarederror',
            'max_depth':6,
            'learning_rate': 0.1,
            'subsample':1.0,
            'colsample_bytree': 1,
            'min_child_weight': 15,
            'n_jobs': -1, 
            }
    num_boost_round = 80


    #print('converting dataframe')
    ## Convert data into xgboost dmatrix format:
    train_df = xgb.DMatrix(df[sub_cols],
                       label=df[target])

    model = xgb.train(
        params=best_params,
        dtrain=train_df,  # Training data
        num_boost_round=num_boost_round,
        )

    del train_df
    gc.collect()
    print('saving model')
    model.save_model(f"3_output/7_uncertainty_boot_seed_{base_seed+i}_model.json")
    
    
    ################### calculating ale #######################################
    # Continuous features by region (feat_var)
    for region in ['warm', 'cool']:
        print(f"{region} {'-' * 15}")
        temp = df.loc[df['div'] == region].copy().reset_index(drop=True)

        print(temp.shape)
        for var in feat_var:
            print(var)
            edges = np.unique([np.quantile(temp[var], q=q, weights=temp['final_weight'], 
                                            method='inverted_cdf') for q in quantiles])
            edges[0] -= PAD
            edges[-1] += PAD

            bin_mid = [(edges[i] + edges[i+1]) / 2 for i in range(len(edges) - 1)]

            out = utils.compute_ale(temp, var, edges, bin_mid, model, 
                              sub_cols, 'final_weight', region)
            out['boot_num'] = base_seed + i
            ale_df = pd.concat([ale_df, out], ignore_index=True)
        del temp
        gc.collect()

In [ ]:
ale_df_all = ale_df.copy()

In [ ]:
ale_df_all.to_parquet('3_output/7_uncertainty_boot_ale.gzip',compression='gzip')

# 4. Optimal condition

In [ ]:
# Parameters
n_bins_around_peak = 4
similarity_threshold = 0.85  # for "similarly high yields"

In [ ]:


def build_range_condition(df, feature, feature_ranges):
    """
    Build OR condition for a feature across all its ranges.
    
    Parameters:
    -----------
    df : DataFrame
        Data to filter
    feature : str
        Feature name (e.g., 'tmin')
    feature_ranges : DataFrame
        Ranges for this feature with 'min_value' and 'max_value' columns
    
    Returns:
    --------
    Boolean Series with OR condition across all ranges
    """
    if len(feature_ranges) == 0:
        # No ranges - return all False
        return pd.Series([False] * len(df), index=df.index)
    
    # Start with first range
    condition = (
        (df[feature] >= feature_ranges.iloc[0]['min_value']) & 
        (df[feature] <= feature_ranges.iloc[0]['max_value'])
    )
    
    # Add OR conditions for remaining ranges
    for i in range(1, len(feature_ranges)):
        condition = condition | (
            (df[feature] >= feature_ranges.iloc[i]['min_value']) & 
            (df[feature] <= feature_ranges.iloc[i]['max_value'])
        )
    
    return condition



In [ ]:
base_seed = 40
boot_opt_candidates = pd.DataFrame()
boot_opt_cond = pd.DataFrame()

for i in np.arange(1,26,1):
    print('#' * 10)
    boot_seed = base_seed + i
    print('seed_num :', boot_seed)
    
    df = pd.read_parquet(f'1_data/bootstrap_seed_{boot_seed}_df.gzip')
    ale_df = ale_df_all.loc[ale_df_all['boot_num'] == boot_seed].copy()
    
    model = xgb.Booster()
    model.load_model(f'3_output/7_uncertainty_boot_seed_{boot_seed}_model.json')
    model.set_param({"device": "cpu"})
    model.set_param({'n_jobs':-1})
    
    all_results = []

    # Process each feature one by one
    for feat in feat_var:
        print(f"\n{'='*80}")
        print(f"PROCESSING FEATURE: {feat}")
        print('='*80)

        for div in ['warm', 'cool']:
            print(f"\n{'-'*60}")
            print(f"Division: {div}")
            print('-'*60)

            # Step 1: Get and filter data
            feature_data = ale_df[
                (ale_df['feat_abv'] == feat) & 
                (ale_df['div'] == div)
            ].copy()

            if len(feature_data) == 0:
                print(f"  No data found")
                continue

            feature_data = feature_data.sort_values('values').reset_index(drop=True)

            if len(feature_data) <= 8:
                print(f"  Not enough data points ({len(feature_data)})")
                continue

            # Filter to 5th-95th percentile [3:-4]
            filtered_data = feature_data.iloc[3:-4].copy()
            filtered_data = filtered_data.reset_index(drop=True)

            print(f"  Bins after filtering: {len(filtered_data)}")

            # Step 2: Find positive peaks
            ale_values = filtered_data['ale'].values
            peaks, _ = find_peaks(ale_values, prominence=np.std(ale_values) * 0.1)
            positive_peaks = [p for p in peaks if ale_values[p] > 0]

            print(f"  Positive peaks found: {len(positive_peaks)}")

            if len(positive_peaks) == 0:
                print(f"  No positive peaks found")
                continue

            # Step 3: Filter isolated/noisy peaks
            valid_peaks = []
            for peak_idx in positive_peaks:
                if not utils.is_isolated_or_noisy_peak(ale_values, peak_idx, 
                                                window=2, 
                                                negative_threshold=0.5, 
                                                neighbor_diff_threshold=1.6):
                    valid_peaks.append(peak_idx)

            print(f"  Valid peaks: {len(valid_peaks)}")

            if len(valid_peaks) == 0:
                print(f"  All peaks are isolated/noisy")
                continue

            # Step 4: Sort peaks by ALE (highest first)
            peak_info_list = []
            for peak_idx in valid_peaks:
                peak_info_list.append({
                    'index': peak_idx,
                    'value': filtered_data.iloc[peak_idx]['values'],
                    'ale': ale_values[peak_idx]
                })
            peak_info_list = sorted(peak_info_list, key=lambda x: x['ale'], reverse=True)

            print(f"\n  Peaks sorted by ALE:")
            for i, p in enumerate(peak_info_list[:5]):  # Show top 5
                print(f"    Peak {i+1}: value={p['value']:.3f}, ALE={p['ale']:.6f}")

            # Step 5: Build optimal ranges
            optimal_ranges = utils.build_optimal_ranges(ale_values, filtered_data, peak_info_list, 
                                                  similarity_threshold=0.8)

            print(f"\n  Optimal ranges created: {len(optimal_ranges)}")

            # Visualize
            if len(optimal_ranges) > 0:
                plt.figure(figsize=(14, 6))
                plt.plot(filtered_data['values'], filtered_data['ale'], 'b-', linewidth=2, label='ALE')

                # Colors for different ranges
                range_colors = ['green', 'orange', 'purple']
                peak_colors = ['red', 'darkred', 'maroon']

                # Plot each range
                for range_idx, range_info in enumerate(optimal_ranges):
                    # Highlight this range
                    range_data = filtered_data[
                        (filtered_data['values'] >= range_info['min_value']) &
                        (filtered_data['values'] <= range_info['max_value'])
                    ]

                    plt.fill_between(
                        range_data['values'], 
                        0, 
                        range_data['ale'],
                        alpha=0.3,
                        color=range_colors[range_idx % len(range_colors)],
                        label=f"Range {range_idx+1}: [{range_info['min_value']:.2f}, {range_info['max_value']:.2f}] ({range_info['n_peaks']} peak(s))"
                    )

                    # Mark peaks in this range
                    for i, (peak_val, peak_ale) in enumerate(zip(range_info['peak_values'], range_info['peak_ales'])):
                        plt.plot(peak_val, peak_ale, 
                                'o', color=peak_colors[range_idx % len(peak_colors)], 
                                markersize=10, 
                                markeredgecolor='black', markeredgewidth=2)

                plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
                plt.xlabel(f'{feat}')
                plt.ylabel('ALE')
                plt.title(f'{feat} - {div}: Optimal Ranges with Hierarchical Peak Inclusion')
                plt.legend(loc='best', fontsize=9)
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

            # Step 6: Store results (only essential columns)
            for range_id, range_info in enumerate(optimal_ranges, start=1):
                result = {
                    'div': div,
                    'feat': feat,
                    'range_id': range_id,
                    'min_value': range_info['min_value'],
                    'max_value': range_info['max_value']
                }
                all_results.append(result)

                print(f"Range {range_id}: [{result['min_value']:.3f}, {result['max_value']:.3f}]")

    # %%
    # Convert to DataFrame
    results_df = pd.DataFrame(all_results)

    opt_candidates = pd.DataFrame()

    for div in ['warm', 'cool']:
        print(f"\nProcessing {div} division")

        # Get ranges for this division
        div_result = results_df.loc[results_df['div'] == div].copy()

        # Start with division filter
        div_condition = (df['div'] == div)

        # Add AND conditions for each feature
        for feat in feat_var:
            # Get ranges for this feature
            feat_ranges = div_result.loc[div_result['feat'] == feat].reset_index(drop=True)
            print(f"  {feat}: {len(feat_ranges)} range(s)")
            if len(feat_ranges) > 0:

                # Build OR condition across all ranges for this feature
                feat_condition = build_range_condition(df, feat, feat_ranges)

                combined = div_condition & feat_condition

                if np.sum(combined)>0:
                    div_condition = combined
                else:
                    print(f'skip {feat} would result in 0 rows')

        # Filter data
        temp_div = df.loc[div_condition].copy()

        # check if no conditions are capture

        print(f"  {div} rows: {len(temp_div)}")

        # Append
        opt_candidates = pd.concat([opt_candidates, temp_div], axis=0, ignore_index=True)

    print(f"\nTotal optimal candidates: {len(opt_candidates)}")
    opt_candidates['boot_num'] = boot_seed
    opt_candidates = opt_candidates.drop_duplicates(subset=['div'] + feat_var)

    for div in ['warm','cool']:
        print('-'*5, div)
        temp = opt_candidates.loc[(opt_candidates['div'] == div) 
                                  & (opt_candidates['boot_num'] == boot_seed)].copy()

        df_temp = df.copy()

        print(temp.shape[0])

        for i in temp.index:
            #print(i)
            for var in feat_var:
                df_temp.loc[df_temp['div'] == div, var] = temp.loc[i,var]

            train = xgb.DMatrix(df_temp[sub_cols])
            pyield = model.predict(train)
            del train
            gc.collect()
            opt_candidates.loc[i, 'pyield'] = np.mean(pyield)
            opt_candidates.loc[i, 'wpyield'] = np.average(pyield, weights= df_temp['final_weight'])

        del temp, df_temp


    warm_cutoff = opt_candidates.loc[opt_candidates['div'] =='warm'].sort_values(by='wpyield', ascending=False).iloc[0,:]
    boot_opt_cond = pd.concat([boot_opt_cond,
                               pd.DataFrame(data=warm_cutoff.to_dict(), index=[boot_seed])],ignore_index=True)
    cool_cutoff = opt_candidates.loc[opt_candidates['div'] =='cool'].sort_values(by='wpyield', ascending=False).iloc[0,:]
    boot_opt_cond = pd.concat([boot_opt_cond,
                               pd.DataFrame(data=cool_cutoff.to_dict(), index=[boot_seed])],ignore_index=True)

    print(boot_opt_cond.loc[boot_opt_cond['boot_num'] == boot_seed][['div']+feat_var])

    boot_opt_candidates = pd.concat([boot_opt_candidates, opt_candidates],axis=0).reset_index(drop=True)

In [ ]:
boot_opt_cond.to_parquet('3_output/7_uncertainty_boot_opt_condition.gzip',compression='gzip')

In [ ]:
boot_opt_candidates.to_parquet('3_output/7_uncertainty_boot_opt_candidates.gzip',compression='gzip')

# 5. Calculating yield loss

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
## some params:
param = {'random':[40,41,42,43],
         'control':['lon','lat','month_cos','month_sin','lac_dim']}

feat_var = ['tmin','tmax_ssrd','rh_am','ag_wind_2m']
target = 'herd_milk_resid'

warm_state = ['CA','AZ','NM','TX','KS','OK','MO','AR','LA','KY','TN','MS','AL','GA','FL','SC','NC','VA']

In [ ]:
target = 'herd_milk_resid'

quantile_y = np.concatenate([np.arange(0.00,0.98,0.02),[0.98,0.99,0.999]])
stress = 'stress'
yloss = 'rel_loss_percent'

In [ ]:
boot_stress_df = pd.DataFrame()
boot_yearly_df = pd.DataFrame()
boot_month_df = pd.DataFrame()
boot_state_prod = pd.DataFrame()

In [ ]:
## Import data:
boot_opt_cond = pd.read_parquet('3_output/7_uncertainty_boot_opt_condition.gzip')

sales = pd.read_csv('1_data/usda_nass_survey_milk_sales_dollars_state_level_2000_2024_annual_level_downloaded_8Aug2025.csv', index_col=0)
sales = sales[['Year','Geo Level','State ANSI','Data Item','Domain','Value']].copy().rename(columns={'Year':'year','Value':'sales','State ANSI':'state_id'})
print(sales['Geo Level'].unique(), sales['Data Item'].unique(), sales['Domain'].unique())
sales = sales.loc[~sales['state_id'].isnull()].copy().reset_index(drop=True)
sales['state_id'] = sales['state_id'].astype(int).astype(str).str.zfill(2)
print('remove D :', sales.loc[sales['sales'] == ' (D)'].shape[0])
sales['sales'] = sales['sales'].replace(",","", regex=True).astype('float64')

ppi = pd.read_csv('1_data/producer_price_index_for_raw_milk_not_seasonally_adjusted_WPU016101.csv', index_col=0)
ppi = ppi.rename(columns={'Year':'year','Value':'ppi'})
ppi = ppi.groupby(['year'])['ppi'].mean().reset_index()
ppi['adj_ppi'] = ppi.loc[ppi['year'] == 2023]['ppi'].values / ppi['ppi']

In [ ]:
for boot_seed in np.arange(41,76,1):
    print('#' * 10)
    print(boot_seed)
    df = pd.read_parquet(f'1_data/bootstrap_seed_{boot_seed}_df.gzip')

    model = xgb.Booster()
    model.load_model(f'3_output/7_uncertainty_boot_seed_{boot_seed}_model.json')
    model.set_param({"device": "cpu"})
    model.set_param({'n_jobs':-1})

    ## opt_condition:
    warm_cutoff = boot_opt_cond.loc[(boot_opt_cond['div'] =='warm') & 
                                   (boot_opt_cond['boot_num'] == boot_seed)].sort_values(by='wpyield', ascending=False).iloc[0,:][feat_var].values
    print('warm opt :', warm_cutoff)
    cool_cutoff = boot_opt_cond.loc[(boot_opt_cond['div'] =='cool') & 
                                   (boot_opt_cond['boot_num'] == boot_seed)].sort_values(by='wpyield', ascending=False).iloc[0,:][feat_var].values
    print('cool opt :', cool_cutoff)

    train_df = xgb.DMatrix(df[sub_cols], df[target])
    df['p_milk'] = model.predict(train_df)
    eval_util_module.evaluate(df[target], df['p_milk'])
    del train_df

    ## remove bad geoid:
    plot_df = df.groupby(['state_abv','GEOID']).apply(lambda g: r2_score(g[target], g['p_milk'])).reset_index(name='r2')
    if plot_df.loc[plot_df['r2'] < 0].shape[0] >0:
        print('!!!!!!!!!!!!! Bad GEOID')
        bad_geoid = plot_df.loc[plot_df['r2'] < 0]['GEOID'].unique()
        df = df.loc[~df['GEOID'].isin(bad_geoid)].copy()

    ## optimum for regional splits:
    train_df = df[np.concatenate([['div'], sub_cols])]
    train_df.loc[train_df['div'] == 'warm',feat_var] = warm_cutoff
    train_df.loc[train_df['div'] =='cool',feat_var] = cool_cutoff
    train_df = xgb.DMatrix(train_df[sub_cols], df['herd_milk_resid'])
    df['p_opt_milk'] = model.predict(train_df)
    del train_df
    print(df.loc[df['p_milk'] > df['p_opt_milk']].shape[0] / df.shape[0])

    ## forcing p_opt_milk to p_milk
    df.loc[df['p_opt_milk'] < df['p_milk'], 'p_opt_milk'] = df.loc[df['p_opt_milk'] < df['p_milk']]['p_milk']

    ## full milk yield predicted values 
    df['opt_milk'] = (df['fitted_herd'] + df['p_opt_milk'])
    df['pred_milk'] = (df['fitted_herd'] + df['p_milk'])

    ## calculating relative yield loss
    df['rel_loss_percent'] = np.nan
    df.loc[:,'rel_loss_percent'] = ((np.exp(df['p_opt_milk']) - np.exp(df['p_milk'])) / np.exp(df['p_opt_milk'])) * 100

    ## defining heat and cold stress
    stress = 'stress'
    df.loc[:,'stress'] = np.nan

    df.loc[(df['div'] == 'warm') & (df['tmin'].round(5) > np.round(warm_cutoff[0],5)) ,stress]='Heat'
    df.loc[(df['div'] == 'warm') & (df['tmin'].round(5) < np.round(warm_cutoff[0],5)),stress] = 'Cold'

    df.loc[(df['div'] == 'cool') & (df['tmin'].round(5) > np.round(cool_cutoff[0],5)) ,stress]='Heat'
    df.loc[(df['div'] == 'cool') & (df['tmin'].round(5) < np.round(cool_cutoff[0],5)) ,stress] ='Cold'

    ## calculating cow-specific test-day relative yield loss
    bin_edge_df = pd.DataFrame()
    stress_df = pd.DataFrame()
    for region in ['warm','cool']:
        for stress_div in ['Cold','Heat']:
            quantile_values = np.quantile(df.loc[(df[stress] == stress_div) & (df['div'] == region)][yloss], q=quantile_y)
            quantile_values = quantile_values*-1

            # Accumulated data points corresponding to each percentile
            N = len(df.loc[(df[stress] == stress_div) & (df['div'] == region)][yloss])

            # Compute bin widths in data points
            bin_sizes = np.diff([0] + quantile_y) * N
            bin_sizes = np.round(bin_sizes).astype(int)

            # Get cumulative bin edges:
            bin_edges = np.concatenate([[0], np.cumsum(bin_sizes)])
            print(bin_edges)

            # For mirrored butterfly:
            if stress_div == 'Cold':
                x_pos = -bin_edges[::-1] / 1e6  # Cold goes left, so reverse
                quantile_values = quantile_values[::-1]
                quantile_labels = [f"{int(q*100) if q<0.998 else '99.99'}%" for q in quantile_y][::-1]
                bin_edges = bin_edges[::-1]
            else:
                x_pos = bin_edges /1e6
                quantile_labels = [f"{int(q*100) if q<0.998 else '99.99'}%" for q in quantile_y]

            data_points1 = np.round(bin_edges /1e6,1)


            stress_df = pd.concat([stress_df, pd.DataFrame(data={'div':region, 'stress':stress_div, 'x_pos':x_pos,
                                                                 'labels':quantile_labels,
                                                                 'value':quantile_values,
                                                                 'n':bin_edges,
                                                                 'data_points':data_points1})], ignore_index=True)
    stress_df['boot_num'] = boot_seed
    boot_stress_df = pd.concat([boot_stress_df, stress_df], ignore_index=True)

    ## saving:
    boot_stress_df.to_parquet('3_output/7_uncertainty_boot_quantile_cow_test_day_loss.gzip', compression='gzip')


    ## yearly loss
    yearly_df = (df.loc[df['year'] < 2024].groupby(['state_abv','GEOID','id','year',stress])[yloss].mean()
                 .reset_index().groupby(['year',stress])[yloss].mean().reset_index())
    yearly_df[yloss] *= -1
    yearly_df['boot_num'] = boot_seed

    boot_yearly_df = pd.concat([boot_yearly_df, yearly_df], ignore_index=True)

    ## saving:
    boot_yearly_df.to_parquet('3_output/7_uncertainty_boot_yearly_loss.gzip', compression='gzip')


    ## monthly loss
    month_df = (df.loc[(df['year'] < 2024)].groupby(['state_abv','GEOID','id','year','month',stress])[yloss].mean()
                .reset_index().groupby(['year','month',stress])[yloss].mean()
                .reset_index().groupby(['month',stress])[yloss].mean().reset_index())
    month_df[yloss] *= -1
    month_df['boot_num'] = boot_seed

    boot_month_df = pd.concat([boot_month_df, month_df], ignore_index=True)

    ## saving:
    boot_month_df.to_parquet('3_output/7_uncertainty_boot_month_loss.gzip', compression='gzip')
    
    ## econ calculation:
    cow_weight = df[['GEOID','id','year','final_weight']].drop_duplicates()
    cow_weight['state_id'] = cow_weight['GEOID'].str[:2]

    ## changing names:
    df['log_opt_milk'] = df['fitted_herd'] + df['p_opt_milk']
    df['log_pred_milk'] = df['fitted_herd'] + df['p_milk']
    df['loss'] = np.nan
    df['opt_milk'] = np.exp(df['log_opt_milk'])
    df['pred_milk'] = np.exp(df['log_pred_milk'])
    df['loss'] = df['opt_milk'] - df['pred_milk']
    stress='stress'
    
    ## sum loss per cow-year-stress
    yearly_loss = (df.groupby(['state_abv','GEOID','id','year',stress])['loss'].sum()
                     .reset_index()
                    )
    ## merging weights
    yearly_loss = pd.merge(yearly_loss, cow_weight, on=['GEOID','id','year'],
                             how='left')
    print('any null ?', yearly_loss.loc[yearly_loss['final_weight'].isnull()].shape[0])
    yearly_loss['Q_loss'] = yearly_loss['loss'] * yearly_loss['final_weight']
    
    ## state-year-stress production loss
    state_loss = yearly_loss.groupby(['state_abv','state_id','year','stress'])['Q_loss'].sum().reset_index()
    
    ## calculating optimal production prediction
    yearly_opt = (df.groupby(['state_abv','GEOID','id','year'])['opt_milk'].sum()
                     .reset_index()
                    )
    yearly_opt = pd.merge(yearly_opt, cow_weight, on=['GEOID','id','year'],
                             how='left')
    yearly_opt['Q_opt'] = yearly_opt['opt_milk'] * yearly_opt['final_weight']
    yearly_opt = yearly_opt.groupby(['state_abv','state_id','year'])['Q_opt'].sum().reset_index()
    
    ## merging:
    state_prod = pd.merge(state_loss, yearly_opt, on=['state_abv','state_id','year'], how='left')
    state_prod['frac_kts'] = state_prod['Q_loss'] / state_prod['Q_opt']
    
    ## calculating r_opt:
    sales_df = pd.merge(sales[['state_id','year','sales']], 
                        state_prod[['state_abv','state_id','year','frac_kts']].groupby(['state_abv','state_id','year'])['frac_kts'].sum().reset_index(),
                        on=['state_id','year'], how='right')
    
    ## merging with sales_df
    sales_df = pd.merge(sales_df, ppi, on=['year'], how='left')
    sales_df['adj_sales'] = (sales_df['sales'] * sales_df['adj_ppi']).astype(float).astype(int)
    sales_df['adj_sales_opt'] = (sales_df['adj_sales'] / (1 - sales_df['frac_kts'])).astype(float).astype(int)
    
    state_prod = state_prod.merge(sales_df.drop(columns=['frac_kts']), on=['state_abv','state_id','year'], how='left')
    state_prod['sales_loss'] = (state_prod['adj_sales_opt'] * state_prod['frac_kts']).astype(float).astype(int)
    state_prod['temp_loss'] = state_prod['adj_sales_opt'] - state_prod['adj_sales']
    state_prod['temp_loss_sum'] = state_prod.groupby(['state_abv','year'])['sales_loss'].transform('sum')
    state_prod['boot_num'] = boot_seed
    
    boot_state_prod = pd.concat([boot_state_prod, state_prod], ignore_index=True)
    boot_state_prod.to_parquet('3_output/7_uncertainty_boot_econ_loss.gzip',compression='gzip')
    
    del cow_weight, yearly_loss, yearly_opt, state_prod, sales_df